<a href="https://colab.research.google.com/github/chaiyawat19/DataScinceLab/blob/main/Lab8_Sentiment_Analysis_with_Naive_Bayse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


from wordcloud import WordCloud
import nltk
import re
import string
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


stop_words = stopwords.words()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/sentimentdata/IMDB5000.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df['sentiment'].value_counts()

In [ ]:
df['review'].str.len().hist()

In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,8))
ax1.hist(df[df['sentiment']=='positive']['review'].str.len())
ax1.set_title( 'Positive Reviews')
ax2.hist(df[df['sentiment']=='negative']['review'].str.len())
ax2.set_title( 'Negative Reviews')

In [ ]:
df.rename(columns={'review':'text'}, inplace = True)
df

In [ ]:
text = " ".join(i for i in df[df['sentiment']=='positive']['review'])
wordcloud = WordCloud( background_color="white").generate(text)

plt.figure( figsize=(15,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title('wordcloud for positive review')
plt.show()

In [ ]:
def cleaning(text):
    # converting to lowercase, removing URL links, special characters, punctuations...
    text = text.lower() # converting to lowercase
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # removing URL links
    text = re.sub(r"\b\d+\b", "", text) # removing number
    text = re.sub(r'<.*?>+', '', text) # removing special characters,
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text) # punctuations
    text = re.sub(r'\n', '', text)
    text = re.sub(r'[’“”…]', '', text)

    #removing emoji:
    emoji_pattern = re.compile(r"["
                           r"\U0001F600-\U0001F64F"  # emoticons
                           r"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           r"\U0001F680-\U0001F6FF"  # transport & map symbols
                           r"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           r"\U00002702-\U000027B0"
                           r"\U000024C2-\U0001F251"
                           r"]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)

   # removing short form:

    text=re.sub(r"isn't",'is not',text)
    text=re.sub(r"he's",'he is',text)
    text=re.sub(r"wasn't",'was not',text)
    text=re.sub(r"there's",'there is',text)
    text=re.sub(r"couldn't",'could not',text)
    text=re.sub(r"won't",'will not',text)
    text=re.sub(r"they're",'they are',text)
    text=re.sub(r"she's",'she is',text)
    text=re.sub(r"There's",'there is',text)
    text=re.sub(r"wouldn't",'would not',text)
    text=re.sub(r"haven't",'have not',text)
    text=re.sub(r"That's",'That is',text)
    text=re.sub(r"you've",'you have',text)
    text=re.sub(r"He's",'He is',text)
    text=re.sub(r"what's",'what is',text)
    text=re.sub(r"weren't",'were not',text)
    text=re.sub(r"we're",'we are',text)
    text=re.sub(r"hasn't",'has not',text)
    text=re.sub(r"you'd",'you would',text)
    text=re.sub(r"shouldn't",'should not',text)
    text=re.sub(r"let's",'let us',text)
    text=re.sub(r"they've",'they have',text)
    text=re.sub(r"You'll",'You will',text)
    text=re.sub(r"i'm",'i am',text)
    text=re.sub(r"we've",'we have',text)
    text=re.sub(r"it's",'it is',text)
    text=re.sub(r"don't",'do not',text)
    text=re.sub(r"that´s",'that is',text)
    text=re.sub(r"I´m",'I am',text)
    text=re.sub(r"it’s",'it is',text)
    text=re.sub(r"she´s",'she is',text)
    text=re.sub(r"he’s'",'he is',text)
    text=re.sub(r'I’m','I am',text)
    text=re.sub(r'I’d','I did',text)
    text=re.sub(r"he’s'",'he is',text)
    text=re.sub(r'there’s','there is',text)


    return text

dt = df['review'].apply(cleaning)
dt.name = 'text'

In [ ]:
dt = pd.DataFrame(dt)
dt['sentiment']=df['sentiment']
dt

In [ ]:
# remove stop word:
dt['no_sw'] = dt['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))

In [ ]:
dt

In [ ]:
#Working with the most Frequent Words:
from collections import Counter
cnt = Counter()
for text in dt["no_sw"].values:
    for word in text.split():
        cnt[word] += 1
cnt.most_common(10)
temp = pd.DataFrame(cnt.most_common(10))
temp.columns=['word', 'count']
temp

In [ ]:
px.bar(temp, x="count", y="word", title='Commmon Words in Text', orientation='h',
             width=700, height=700)

In [ ]:
# Remove the most frequent words:
FREQWORDS = set([w for (w, wc) in cnt.most_common(10)])
def remove_freqwords(text):
    """custom function to remove the frequent words"""
    return " ".join([word for word in str(text).split() if word not in FREQWORDS])
dt["wo_stopfreq"] = dt["no_sw"].apply(lambda text: remove_freqwords(text))
dt.head()

In [ ]:
dt['no_sw'].loc[0]

In [ ]:
dt['wo_stopfreq'].loc[0]

In [ ]:
dt.columns

In [ ]:
nb=dt.drop(columns=['text', 'wo_stopfreq'])
nb.columns=['sentiment','no_sw']
nb.sentiment = [0 if each == "negative" else 1 for each in nb.sentiment]
nb

In [ ]:
#separating the data and label
X = nb['no_sw'].values
Y = nb['sentiment'].values

In [ ]:
print(X)

In [ ]:
print(Y)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# converting the textual data to numerical data
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [ ]:
print(X)

Naive Bayse Sklearn provides 5 types of Naive Bayes :
*  GaussianNB works for continuous features
*  CategoricalNB works for categorical features
*  BernoulliNB works for binary features
*  MultinomialNB works for multinomial features
*  ComplementNB work for multinomial features/ imbalance/ text classification


In [ ]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.model_selection import train_test_split
# Create NB classifer object
NBclassifier = BernoulliNB()

# Train NB Classifer
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify=Y, random_state=2)

NBclassifier.fit(X_train, Y_train)
y_predict = NBclassifier.predict(X_test)

In [ ]:
from sklearn import metrics
print("Accuracy:",metrics.accuracy_score(Y_test,y_predict))

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix

print(classification_report(Y_test,y_predict))